**Simulação Computacional - Projecto Mecânico - Aula Prática 6**

<br>

# Elemento de Viga (*BEAM*)

<br>

Capítulo 4 (Logan 5th) vamos agora introduzir último elemento unidimencional desta cadeira de simulação computacional. Em análise estrutural falamos agora do elemento viga.

Ter em atenção que por oposição ao elemento barra, o elemento de viga simples tem 4 graus de liberdade (no espaço do elemento), dois por nó, onde se consideram deslocamentos perpendiculares ao eixo do elemento e a possibilidade de ter um momento rotacional em cada nó, deformações axiais são ignoradas no elemento de viga simples, diz-se que se consideram apenas a deformações de dobra seguindo a teoria das vigas proposta por Euler-Bernouli. A determinação da matriz de rigidez encontra-se descrita no livro das páginas 152 a 155, sendo que aqui colocalmos a sua forma final.

<br>

$$
\hat{k} = \frac{EI}{L^3}
\begin{pmatrix}
12 & 6L & -12 & 6L \\
6L & 4L^2 & -6L & 2L^2 \\
-12 & -6L & 12 & -6L \\
6L & 2L^2 & -6L & 4L^2 \\
\end{pmatrix}
$$

<br>

onde $I$ é o momento de inércia principal no eixo dos $\hat{z}$ que é perpendicular a $\hat{x}$ e $\hat{y}$. Assim a matriz de rididez em coordenadas locais do elemento relaciona:

<br>

$$
\begin{pmatrix}
\hat{f_{1y}} \\
\hat{m_1} \\
\hat{f_{2y}} \\
\hat{m_2} \\
\end{pmatrix}
= \frac{EI}{L^3}
\begin{pmatrix}
12 & 6L & -12 & 6L \\
6L & 4L^2 & -6L & 2L^2 \\
-12 & -6L & 12 & -6L \\
6L & 2L^2 & -6L & 4L^2 \\
\end{pmatrix}
\begin{pmatrix}
\hat{d_{1y}} \\
\hat{\phi_1} \\
\hat{d_{2y}} \\
\hat{\phi_2} \\
\end{pmatrix}
$$

<br>

---

Exemplo da resolução de um problema de vigas. Considera a viga da próxima figura:

![Imagem:](https://drive.google.com/uc?export=view&id=1pxR7uad9BI600MAPfb-VIo0krRG75pgU)

<br>

onde temos uma viga que se assume ter um EI constante ao longo dela, e onde uma força de 1000 N e um momento de 1000 N/m foi aplicado exactamente a meio da viga. Á esquerda temos um suporte fixo, á direita temos um suporte num pino.

Vamos obter a solução do problema global, sendo que aqui por o elemento estar alinhado com os eixos globais do problema, não necessitamos de transformações adicicionais, assim...

In [ ]:
import sympy
from sympy import symbols, Matrix, sqrt, cos, acos, eye, zeros, solve


# 1. Definição das Propriedades da Viga e Parâmetros Geométricos
# E: Módulo de Young (Elasticidade)
# I: Momento de Inércia da secção transversal
# L_e: Comprimento de cada elemento de viga

E, I, L_e = symbols('E I L_e')

# Nó 1: Suporte fixo (esquerda)
# Nó 2: Meio da viga (aplicação da carga e momento)
# Nó 3: Suporte de pino (direita)

# 2. Matriz de Rigidez do Elemento de Viga (Euler-Bernoulli)
# A matriz de rigidez para um elemento de viga com 4 graus de liberdade (deslocamento vertical e rotação em cada nó).
# A ordem dos graus de liberdade é [d1, phi1, d2, phi2] ou [d2, phi2, d3, phi3].

k_element = (E * I / L_e**3) * sympy.Matrix([
    [12, 6*L_e, -12, 6*L_e],
    [6*L_e, 4*L_e**2, -6*L_e, 2*L_e**2],
    [-12, -6*L_e, 12, -6*L_e],
    [6*L_e, 2*L_e**2, -6*L_e, 4*L_e**2]
])

print("Matriz de Rigidez de um Elemento (simbólica):")
display(k_element)

# 3. Montagem da Matriz de Rigidez Global
# A viga tem 3 nós e cada nó tem 2 graus de liberdade (deslocamento vertical e rotação).
# Total de graus de liberdade globais = 3 nós * 2 GDL/nó = 6.
# Os GDL globais são: [d1, phi1, d2, phi2, d3, phi3]

K_global = sympy.zeros(6, 6)

# Elemento 1 conecta o Nó 1 e o Nó 2. Os GDLs são [d1, phi1, d2, phi2]
# Mapeamento: d1 -> 0, phi1 -> 1, d2 -> 2, phi2 -> 3
indices_e1 = [0, 1, 2, 3]
for i_local, i_global in enumerate(indices_e1):
    for j_local, j_global in enumerate(indices_e1):
        K_global[i_global, j_global] += k_element[i_local, j_local]

# CElemento 2 conecta o Nó 2 e o Nó 3. Os GDLs são [d2, phi2, d3, phi3]
# Mapeamento: d2 -> 2, phi2 -> 3, d3 -> 4, phi3 -> 5
indices_e2 = [2, 3, 4, 5]
for i_local, i_global in enumerate(indices_e2):
    for j_local, j_global in enumerate(indices_e2):
        K_global[i_global, j_global] += k_element[i_local, j_local]

print("\nMatriz de Rigidez Global (simbólica):")
display(K_global)

# 4. Vetor de Forças Globais
# As cargas são aplicadas no Nó 2.
# Força vertical F_y = -1000 N (negativo porque é para baixo)
# Momento M_z = 1000 N/m (positivo por convenção anti-horária)
# Os GDLs são: [v1, theta1, v2, theta2, v3, theta3]
# O Nó 2 corresponde aos GDLs de índice 2 (v2) e 3 (theta2).

F_global = sympy.zeros(6, 1)
F_global[2] = -1000 # Carga vertical no Nó 2
F_global[3] = 1000  # Momento no Nó 2

print("\nVetor de Forças e Momentos Globais:")
display(F_global)

# 5. Aplicação das Condições de Fronteira (encontrar o problema reduzido)
# Nó 1 (fixo): d1 = 0 (GDL 0), phi1 = 0 (GDL 1)
# Nó 3 (pino): d3 = 0 (GDL 4)
# As linhas e colunas correspondentes aos GDLs restritos são removidas.
# Os GDLs conhecidos são: 0, 1, 4.
restrained_dofs = [0, 1, 4]
free_dofs = [dof for dof in range(6) if dof not in restrained_dofs]

K_reduced = K_global.extract(free_dofs, free_dofs)
F_reduced = F_global.extract(free_dofs, [0]) # Extrair apenas a coluna de cargas

print("\nMatriz de Rigidez Reduzida (após BCs):")
display(K_reduced)
print("\nVetor de Cargas Reduzido (após BCs):")
display(F_reduced)

# 6. Resolução para Deslocamentos e Rotações Desconhecidos
# Resolvemos o sistema de equações K_reduzido * U_reduzido = F_reduzido.
# Para resolver numericamente, precisamos de valores para E, I e L_e

# 7. Tal como anteriormente, depois de encontrados os deslocamentos desconhecidos
# podemos resolver para encontrar as forças desconhecidas.


Matriz de Rigidez de um Elemento (simbólica):


Matrix([
[ 12*E*I/L_e**3,  6*E*I/L_e**2, -12*E*I/L_e**3,  6*E*I/L_e**2],
[  6*E*I/L_e**2,     4*E*I/L_e,  -6*E*I/L_e**2,     2*E*I/L_e],
[-12*E*I/L_e**3, -6*E*I/L_e**2,  12*E*I/L_e**3, -6*E*I/L_e**2],
[  6*E*I/L_e**2,     2*E*I/L_e,  -6*E*I/L_e**2,     4*E*I/L_e]])


Matriz de Rigidez Global (simbólica):


Matrix([
[ 12*E*I/L_e**3,  6*E*I/L_e**2, -12*E*I/L_e**3,  6*E*I/L_e**2,              0,             0],
[  6*E*I/L_e**2,     4*E*I/L_e,  -6*E*I/L_e**2,     2*E*I/L_e,              0,             0],
[-12*E*I/L_e**3, -6*E*I/L_e**2,  24*E*I/L_e**3,             0, -12*E*I/L_e**3,  6*E*I/L_e**2],
[  6*E*I/L_e**2,     2*E*I/L_e,              0,     8*E*I/L_e,  -6*E*I/L_e**2,     2*E*I/L_e],
[             0,             0, -12*E*I/L_e**3, -6*E*I/L_e**2,  12*E*I/L_e**3, -6*E*I/L_e**2],
[             0,             0,   6*E*I/L_e**2,     2*E*I/L_e,  -6*E*I/L_e**2,     4*E*I/L_e]])


Vetor de Forças e Momentos Globais:


Matrix([
[    0],
[    0],
[-1000],
[ 1000],
[    0],
[    0]])


Matriz de Rigidez Reduzida (após BCs):


Matrix([
[24*E*I/L_e**3,         0, 6*E*I/L_e**2],
[            0, 8*E*I/L_e,    2*E*I/L_e],
[ 6*E*I/L_e**2, 2*E*I/L_e,    4*E*I/L_e]])


Vetor de Cargas Reduzido (após BCs):


Matrix([
[-1000],
[ 1000],
[    0]])

---

Um segundo exemplo:

![Imagem:](https://drive.google.com/uc?export=view&id=1ujQDwsL_VM7rXhVr8btF0fP75pZ9HPEg)

Na imagem temos uma *cantilever* que tem uma força externa aplicada no nó 1, $P$, tendo um $EI$ constante e um comprimento igual a $2L$ estando fixa no nó 3, e apoiada num rolamento no nó 2. A figura já contém a discretização do problema. Vamos agora determinar os deslocamentos nodaise e rotações, as reações e as forças de cizalhamento assim como o diagrama dos momentos de torção.



In [ ]:
import sympy
from sympy import symbols, Matrix, zeros
from IPython.display import display

# 1. Definição das Propriedades da Viga e Parâmetros Geométricos
# Comentário: Definir as variáveis simbólicas para o módulo de Young (E), momento de inércia (I),
#             comprimento de cada elemento (L) e a força aplicada (P).
#             A viga total tem comprimento 2L, dividida em dois elementos de comprimento L.
E, I, L, P = symbols('E I L P')

# Comentário: A matriz de rigidez de um elemento de viga, baseada na teoria de Euler-Bernoulli.
#             Os graus de liberdade são [deslocamento_y1, rotação_z1, deslocamento_y2, rotação_z2].
k_element = (E * I / L**3) * Matrix([
    [12, 6*L, -12, 6*L],
    [6*L, 4*L**2, -6*L, 2*L**2],
    [-12, -6*L, 12, -6*L],
    [6*L, 2*L**2, -6*L, 4*L**2]
])

print("Matriz de Rigidez de um Elemento (simbólica):")
display(k_element)
# Comentário: Esta matriz é usada para ambos os elementos, pois ambos têm o mesmo EI e L.

# 2. Montagem da Matriz de Rigidez Global
# Comentário: A viga tem 3 nós e cada nó tem 2 graus de liberdade (deslocamento vertical e rotação).
#             Total de graus de liberdade globais = 3 nós * 2 GDL/nó = 6.
#             A ordem dos GDLs globais é: [d1, phi1, d2, phi2, d3, phi3].
K_global = zeros(6, 6)

# Comentário: Elemento 1 conecta o Nó 1 e o Nó 2. Os GDLs locais [d1, phi1, d2, phi2] mapeiam
#             para os GDLs globais [0, 1, 2, 3].
indices_e1 = [0, 1, 2, 3]
for i_local, i_global in enumerate(indices_e1):
    for j_local, j_global in enumerate(indices_e1):
        K_global[i_global, j_global] += k_element[i_local, j_local]

# Comentário: Elemento 2 conecta o Nó 2 e o Nó 3. Os GDLs locais [d2, phi2, d3, phi3] mapeiam
#             para os GDLs globais [2, 3, 4, 5].
indices_e2 = [2, 3, 4, 5]
for i_local, i_global in enumerate(indices_e2):
    for j_local, j_global in enumerate(indices_e2):
        K_global[i_global, j_global] += k_element[i_local, j_local]

print("\nMatriz de Rigidez Global (simbólica):")
display(K_global)

# 3. Vetor de Forças Globais
# Comentário: Definir o vetor de forças globais. Uma força P é aplicada para baixo no Nó 1.
#             Portanto, F_global[0] = -P. Não há momentos ou outras forças aplicadas diretamente.
F_global = zeros(6, 1)
F_global[0] = -P  # Força P aplicada no Nó 1, para baixo (negativo)

print("\nVetor de Forças e Momentos Globais (após aplicação de carga externa):")
display(F_global)

# 4. Aplicação das Condições de Fronteira
# Comentário: Identificar os graus de liberdade restritos pelas condições de apoio.
#             Nó 2 (apoio de rolete): d2 = 0 (GDL 2). Permite rotação.
#             Nó 3 (engaste): d3 = 0 (GDL 4), phi3 = 0 (GDL 5). Não permite deslocamento nem rotação.
restrained_dofs = [2, 4, 5]
# Comentário: Os GDLs livres são os que não estão restritos.
free_dofs = [dof for dof in range(6) if dof not in restrained_dofs] # GDLs: 0 (d1), 1 (phi1), 3 (phi2)

# Comentário: Extrair as submatrizes K_reduzido e F_reduzido correspondentes aos GDLs livres.
K_reduced = K_global.extract(free_dofs, free_dofs)
F_reduced = F_global.extract(free_dofs, [0]) # Extrair apenas a coluna de cargas

print("\nMatriz de Rigidez Reduzida (após aplicação das Condições de Fronteira):")
display(K_reduced)
print("\nVetor de Cargas Reduzido (após aplicação das Condições de Fronteira):")
display(F_reduced)

# 5. Resolução para Deslocamentos e Rotações Desconhecidos
# Comentário: Resolver o sistema de equações K_reduzido * U_reduzido = F_reduzido para encontrar os deslocamentos e rotações desconhecidos.
U_reduced = K_reduced.LUsolve(F_reduced)

# Comentário: Exibir os deslocamentos e rotações calculados para os GDLs livres.
print(f"\nDeslocamentos e Rotações Desconhecidos (GDLs {free_dofs} -> d1, phi1, phi2):")
display(U_reduced)

# 6. Reconstrução do Vetor de Deslocamentos Global Completo
# Comentário: Criar o vetor de deslocamentos global completo, preenchendo com os valores calculados
#             e com zero para os GDLs restritos.
U_global = zeros(6, 1)
for i, dof in enumerate(free_dofs):
    U_global[dof] = U_reduced[i]

# Comentário: Exibir o vetor de deslocamentos global completo.
print("\nVetor de Deslocamentos Global Completo:")
display(U_global)

# 7. Cálculo das Reações nos Apoios
# Comentário: Calcular as forças de reação nos GDLs restritos usando a matriz de rigidez global
#             e o vetor de deslocamentos global. As reações são a diferença entre as forças internas
#             e as forças externas aplicadas nos GDLs restritos.
R_global = K_global * U_global - F_global

# Comentário: Exibir as reações nos apoios.
print("\nReações nos Apoios (Forças e Momentos de Reação nos GDLs restritos):")
# Os GDLs restritos são [2, 4, 5]
print(f"Reação em d2 (Nó 2, vertical): {R_global[2]}")
print(f"Reação em d3 (Nó 3, vertical): {R_global[4]}")
print(f"Reação em phi3 (Nó 3, momento): {R_global[5]}")

# 8. Cálculo das Forças Internas (Forças de Cisalhamento e Momentos Fletores)
# Comentário: Calcular as forças internas (cisalhamento e momento fletor) em cada elemento.

# Para o Elemento 1 (entre Nó 1 e Nó 2)
# Comentário: Extrair os deslocamentos e rotações dos nós do Elemento 1 do vetor U_global.
u_e1 = U_global.extract([0, 1, 2, 3], [0]) # [d1, phi1, d2, phi2]
# Comentário: Calcular as forças nodais locais para o Elemento 1.
f_e1 = k_element * u_e1

print("\nForças Internas no Elemento 1 (do Nó 1 para o Nó 2):")
print(f"Força de Cisalhamento no Nó 1 do Elemento 1 (V1): {f_e1[0]}")
print(f"Momento Fletor no Nó 1 do Elemento 1 (M1): {f_e1[1]}")
print(f"Força de Cisalhamento no Nó 2 do Elemento 1 (V2): {f_e1[2]}")
print(f"Momento Fletor no Nó 2 do Elemento 1 (M2): {f_e1[3]}")

# Para o Elemento 2 (entre Nó 2 e Nó 3)
# Comentário: Extrair os deslocamentos e rotações dos nós do Elemento 2 do vetor U_global.
u_e2 = U_global.extract([2, 3, 4, 5], [0]) # [d2, phi2, d3, phi3]
# Comentário: Calcular as forças nodais locais para o Elemento 2.
f_e2 = k_element * u_e2

print("\nForças Internas no Elemento 2 (do Nó 2 para o Nó 3):")
print(f"Força de Cisalhamento no Nó 2 do Elemento 2 (V2): {f_e2[0]}")
print(f"Momento Fletor no Nó 2 do Elemento 2 (M2): {f_e2[1]}")
print(f"Força de Cisalhamento no Nó 3 do Elemento 2 (V3): {f_e2[2]}")
print(f"Momento Fletor no Nó 3 do Elemento 2 (M3): {f_e2[3]}")

# Comentário: Para plotar os diagramas de esforço transverso e momento fletor,
#             seria necessário substituir valores numéricos para E, I, L e P,
#             e usar uma biblioteca de plotagem. Os valores calculados acima
#             são as forças e momentos nos nós dos elementos.

Matriz de Rigidez de um Elemento (simbólica):


Matrix([
[ 12*E*I/L**3,  6*E*I/L**2, -12*E*I/L**3,  6*E*I/L**2],
[  6*E*I/L**2,     4*E*I/L,  -6*E*I/L**2,     2*E*I/L],
[-12*E*I/L**3, -6*E*I/L**2,  12*E*I/L**3, -6*E*I/L**2],
[  6*E*I/L**2,     2*E*I/L,  -6*E*I/L**2,     4*E*I/L]])


Matriz de Rigidez Global (simbólica):


Matrix([
[ 12*E*I/L**3,  6*E*I/L**2, -12*E*I/L**3,  6*E*I/L**2,            0,           0],
[  6*E*I/L**2,     4*E*I/L,  -6*E*I/L**2,     2*E*I/L,            0,           0],
[-12*E*I/L**3, -6*E*I/L**2,  24*E*I/L**3,           0, -12*E*I/L**3,  6*E*I/L**2],
[  6*E*I/L**2,     2*E*I/L,            0,     8*E*I/L,  -6*E*I/L**2,     2*E*I/L],
[           0,           0, -12*E*I/L**3, -6*E*I/L**2,  12*E*I/L**3, -6*E*I/L**2],
[           0,           0,   6*E*I/L**2,     2*E*I/L,  -6*E*I/L**2,     4*E*I/L]])


Vetor de Forças e Momentos Globais (após aplicação de carga externa):


...


Deslocamentos e Rotações Desconhecidos (GDLs [0, 1, 3] -> d1, phi1, phi2):


Matrix([
[-7*L**3*P/(12*E*I)],
[  3*L**2*P/(4*E*I)],
[    L**2*P/(4*E*I)]])


Vetor de Deslocamentos Global Completo:


Matrix([
[-7*L**3*P/(12*E*I)],
[  3*L**2*P/(4*E*I)],
[                 0],
[    L**2*P/(4*E*I)],
[                 0],
[                 0]])


Reações nos Apoios (Forças e Momentos de Reação nos GDLs restritos):
Reação em d2 (Nó 2, vertical): 5*P/2
Reação em d3 (Nó 3, vertical): -3*P/2
Reação em phi3 (Nó 3, momento): L*P/2

Forças Internas no Elemento 1 (do Nó 1 para o Nó 2):
Força de Cisalhamento no Nó 1 do Elemento 1 (V1): -P
Momento Fletor no Nó 1 do Elemento 1 (M1): 0
Força de Cisalhamento no Nó 2 do Elemento 1 (V2): P
Momento Fletor no Nó 2 do Elemento 1 (M2): -L*P

Forças Internas no Elemento 2 (do Nó 2 para o Nó 3):
Força de Cisalhamento no Nó 2 do Elemento 2 (V2): 3*P/2
Momento Fletor no Nó 2 do Elemento 2 (M2): L*P
Força de Cisalhamento no Nó 3 do Elemento 2 (V3): -3*P/2
Momento Fletor no Nó 3 do Elemento 2 (M3): L*P/2


## Exercício de Consolidação

Convido-vos a resolverem o exercício P4-8 da página 204 do *FEM First Course - Logan 5th Edition*.

![Imagem:](https://drive.google.com/uc?export=view&id=1uyq-WSmJNhdHnYV1O5CUEZT20SdaSsz0)

Para ambos os problemas determina os deslocamentos e os momentos em cada nó, assim como as forças em cada elemento e as reações.

---